# Assignment 3: Complete DFA and FST Pipeline

This notebook integrates the DFA and FST to process the entire noun corpus from `brown_nouns.txt`.
It runs the DFA check on all words, then processes accepted words with the FST, and writes the output.

In [1]:
import os
import re

# --- DFA CLASS ---
class DFA:
    def __init__(self):
        self.states = {'q0', 'q1', 'q2'}
        self.start_state = 'q0'
        self.accept_states = {'q1'}
        self.current_state = self.start_state

    def transition(self, char):
        if self.current_state == 'q2':
            return 'q2'
        if self.current_state == 'q0':
            if char.islower() and char.isalpha():
                self.current_state = 'q1'
            else:
                self.current_state = 'q2'
        elif self.current_state == 'q1':
            if char.islower() and char.isalpha():
                self.current_state = 'q1'
            else:
                self.current_state = 'q2'
        return self.current_state

    def is_accepted(self, word):
        self.reset()
        if not word:
            return False
        for char in word:
            self.transition(char)
        return self.current_state in self.accept_states

    def reset(self):
        self.current_state = self.start_state

# --- FST CLASS ---
class FST:
    def __init__(self):
        self.states = {
            'q_start', 'q_vow', 'q_con', 'q_s', 'q_c', 'q_e_insert', 
            'q_con_y', 'q_vow_y', 'q_Y_i', 'q_Y_ie', 'q_es', 'q_PL_mid', 
            'q_SG', 'q_PL'
        }
        self.start_state = 'q_start'
        self.accept_states = {'q_SG', 'q_PL'}
        self.vowels = set('aeiou')
        self.consonants = set('bcdfghjklmnpqrstvwxz')
        self.transitions = {}
        self.build_transitions()
        self.lexicon = set()

    def add_transition(self, from_state, input_sym, to_state, output_sym):
        if (from_state, input_sym) not in self.transitions:
            self.transitions[(from_state, input_sym)] = []
        self.transitions[(from_state, input_sym)].append((to_state, output_sym))

    def build_transitions(self):
        root_states = {
            'q_start', 'q_vow', 'q_con', 'q_s', 'q_c', 'q_e_insert', 
            'q_con_y', 'q_vow_y'
        }
        for state in root_states:
            for char in 'abcdefghijklmnopqrstuvwxyz':
                if char == 's':
                    next_state = 'q_s'
                elif char in ('z', 'x'):
                    next_state = 'q_e_insert'
                elif char == 'c':
                    next_state = 'q_c'
                elif char == 'h':
                    if state == 'q_c':
                        next_state = 'q_e_insert'
                    elif state == 'q_s':
                        next_state = 'q_e_insert'
                    else:
                        next_state = 'q_con'
                elif char == 'y':
                    if state in ('q_vow', 'q_vow_y'):
                        next_state = 'q_vow_y'
                    else:
                        next_state = 'q_con_y'
                elif char in self.vowels:
                    next_state = 'q_vow'
                else:
                    next_state = 'q_con'
                self.add_transition(state, char, next_state, char)
        
        for state in ('q_s', 'q_e_insert'):
            self.add_transition(state, 'e', 'q_es', '')
        self.add_transition('q_es', 's', 'q_PL_mid', '')
        
        consonant_endings = ('q_con', 'q_s', 'q_c', 'q_e_insert')
        for state in consonant_endings:
            self.add_transition(state, 'i', 'q_Y_i', 'y')
        self.add_transition('q_Y_i', 'e', 'q_Y_ie', '')
        self.add_transition('q_Y_ie', 's', 'q_PL_mid', '')
        
        for state in ('q_vow', 'q_vow_y', 'q_con', 'q_c'):
            self.add_transition(state, 's', 'q_PL_mid', '')
            
        for state in root_states:
            self.add_transition(state, None, 'q_SG', '+N+SG')
        self.add_transition('q_PL_mid', None, 'q_PL', '+N+PL')

    def load_lexicon(self, corpus_path):
        with open(corpus_path, "r", encoding="utf-8") as f:
            raw_words = [line.strip() for line in f if line.strip()]
        dfa_pattern = re.compile(r"^[a-z]+$")
        valid_words = [w for w in raw_words if dfa_pattern.match(w)]
        self.lexicon = set()
        for w in valid_words:
            self.lexicon.add(self._get_heuristic_root(w))
        print(f"Loaded {len(self.lexicon)} singular roots.")

    def _get_heuristic_root(self, word):
        if word.endswith('ss'):
            return word
        if word.endswith('ies'):
            root = word[:-3] + 'y'
            if len(root) >= 2 and root[-2] in self.consonants:
                return root
        if word.endswith('es'):
            for suffix in ['ses', 'zes', 'xes', 'ches', 'shes']:
                if word.endswith(suffix):
                    return word[:-2]
            if len(word) > 2:
                return word[:-1]
        if word.endswith('s'):
            return word[:-1]
        return word

    def transduce(self, word):
        results = []
        queue = [('q_start', 0, "")]
        while queue:
            state, idx, out = queue.pop(0)
            if idx == len(word):
                if (state, None) in self.transitions:
                    for next_state, out_sym in self.transitions[(state, None)]:
                        if next_state in self.accept_states:
                            results.append((out, next_state))
            if idx < len(word):
                char = word[idx]
                if (state, char) in self.transitions:
                    for next_state, out_sym in self.transitions[(state, char)]:
                        queue.append((next_state, idx + 1, out + out_sym))
        return results

    def analyze(self, word):
        candidates = self.transduce(word)
        valid_analyses = []
        for root, accept_state in candidates:
            if root in self.lexicon:
                suffix = "+N+SG" if accept_state == 'q_SG' else "+N+PL"
                valid_analyses.append(f"{root}{suffix}")
        if not valid_analyses:
            return "Invalid Word"
        if len(valid_analyses) > 1:
            plurals = [a for a in valid_analyses if a.endswith("+PL")]
            if plurals and word.endswith('s') and not word.endswith('ss'):
                singulars = [a for a in valid_analyses if a.endswith("+SG")]
                exact_singular = [s for s in singulars if s.split("+")[0] == word]
                if exact_singular:
                    return exact_singular[0]
                return plurals[0]
            return valid_analyses[0]
        return valid_analyses[0]

In [2]:
corpus_path = "brown_nouns.txt"
dfa = DFA()
fst = FST()
fst.load_lexicon(corpus_path)

with open(corpus_path, "r", encoding="utf-8") as f:
    raw_words = [line.strip() for line in f if line.strip()]

print(f"Total corpus words: {len(raw_words)}")
unique_words = sorted(list(set(raw_words)))
print(f"Total unique words: {len(unique_words)}")

output_lines = []
dfa_accepted = 0
fst_valid = 0
fst_invalid = 0

for w in unique_words:
    if dfa.is_accepted(w):
        dfa_accepted += 1
        analysis = fst.analyze(w)
        if analysis != "Invalid Word":
            fst_valid += 1
            output_lines.append(f"{w} = {analysis}")
        else:
            fst_invalid += 1
            output_lines.append(f"{w} = Invalid Word")
    else:
        output_lines.append(f"{w} = Not Accepted (DFA)")

with open("output.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(output_lines) + "\n")

print(f"\n--- STATISTICS ---")
print(f"Total Unique Words: {len(unique_words)}")
print(f"Accepted by DFA: {dfa_accepted} ({dfa_accepted/len(unique_words)*100:.2f}%)")
print(f"Successfully Inflected by FST: {fst_valid}")
print(f"Spelling Violations (Invalid): {fst_invalid}")

print(f"\n--- SAMPLES ---")
for w in ["fox", "foxes", "foxs", "watches", "watchs", "tries", "trys", "bags", "bages", "gases", "classes"]:
    print(f"{w} -> DFA: {dfa.is_accepted(w)} | FST: {fst.analyze(w)}")

Loaded 12821 singular roots.
Total corpus words: 202793
Total unique words: 19287

--- STATISTICS ---
Total Unique Words: 19287
Accepted by DFA: 17053 (88.42%)
Successfully Inflected by FST: 17050
Spelling Violations (Invalid): 3

--- SAMPLES ---
fox -> DFA: True | FST: fox+N+SG
foxes -> DFA: True | FST: fox+N+PL
foxs -> DFA: True | FST: Invalid Word
watches -> DFA: True | FST: watch+N+PL
watchs -> DFA: True | FST: Invalid Word
tries -> DFA: True | FST: try+N+PL
trys -> DFA: True | FST: Invalid Word
bags -> DFA: True | FST: bag+N+PL
bages -> DFA: True | FST: Invalid Word
gases -> DFA: True | FST: gas+N+PL
classes -> DFA: True | FST: class+N+PL
